# Final Butterfly Dataset — augmentation and Drive hosting

**Author**: Nikolas Antoniou

**Supervisor**: Dr. Demetris Trihinas

**Project**: Butterfly Classification (Cyprus)

`butterfly_images.csv` is 15,088 images and every one of them comes from
iNaturalist. That is a single photographic regime — one platform, one community,
one set of framing conventions — so a classifier trained on it learns some of
iNaturalist along with the butterflies, and it has no way to tell us which.

This notebook adds a second and a third source, keeps them labelled so the
difference can be measured, and rehosts everything on Google Drive so the
resulting table is portable:

| Source | What it is | Licence |
|---|---|---|
| `cyprus_inat`, `global_inat` | the existing iNaturalist dataset | open, attributed |
| `gbif` | ~250/species from GBIF, iNaturalist records excluded | open, attributed |
| `web_scrape` | last-resort top-up for the Cyprus endemics | **unknown** |

The deliverable is `Data/ButterfliesOnly/final_butterfly.csv`.

___________

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# The pipeline lives in src/insects; the notebook imports it rather than
# re-implementing any of it.
sys.path.insert(0, str(Path.cwd().parent / "src"))
from insects import assemble, config, dedup, drive  # noqa: E402

pd.set_option("display.width", 140)

## 2. What each source contributed

The two harvests are already on disk (`python -m insects.gbif --harvest`,
`python -m insects.webscrape --harvest`). This is what they returned.

In [ ]:
gbif = pd.read_csv(config.GBIF_MEDIA_CSV)
web = pd.read_csv(config.WEB_MEDIA_CSV)
inat = pd.read_csv(config.BUTTERFLY_IMAGES_CSV)

print(f"iNaturalist : {len(inat):>6,} images")
print(f"GBIF        : {len(gbif):>6,} images")
print(f"web scrape  : {len(web):>6,} images")

### 2.1 GBIF licences

GBIF's largest non-iNaturalist publisher releases under **NoDerivatives**, which
is usable as training input but may not be redistributed as a dataset. Those
rows are admitted and flagged (`redistributable = False`) rather than dropped —
dropping them would have cost ~80% of the harvest, and hiding the distinction
would make the datasheet wrong.

In [ ]:
licences = gbif["license_code"].value_counts().rename("images").to_frame()
licences["share_%"] = (100 * licences["images"] / len(gbif)).round(1)
licences["redistributable"] = ~licences.index.isin(config.TRAINING_ONLY_LICENCES)
licences

### 2.2 Where GBIF ran dry

The species GBIF could not fill are the Cyprus endemics — *Glaucopsyche paphos*,
*Hipparchia cypriensis*, *Maniola cypricola*. Outside iNaturalist there is very
little openly licensed material of them anywhere, which is precisely why they
were the starved classes in the first place. The web scrape exists for these.

In [ ]:
target = config.TOPUP_PER_SPECIES
per_species = pd.DataFrame({
    "gbif": gbif["label_species"].value_counts(),
    "web_scrape": web["label_species"].value_counts(),
}).reindex(sorted(inat["label_species"].unique())).fillna(0).astype(int)
per_species["total_new"] = per_species.sum(axis=1)
per_species["short_of_target"] = (target - per_species["total_new"]).clip(lower=0)
per_species.sort_values("short_of_target", ascending=False).head(10)

## 3. Assemble

`assemble.assemble()` concatenates the three sources into one schema, takes
family and genus from the audited iNaturalist taxonomy (so all three sources
agree), derives the licence flags from `config`, then opens every file on disk to
record its real dimensions and perceptual hash.

It then caps every image at 800px on the long side. GBIF hands back the
photographer's original — up to 4000px — where iNaturalist serves a ~500px
"medium", and that difference is worth removing for two reasons: it takes the
Drive upload from ~5.5 GB to ~2 GB, and it stops *resolution* from being a
give-away signal that would let a model separate the sources instead of the
species. The `width` and `height` columns keep the published dimensions, so the
comparison in section 4 is still made on the images as they were taken.

This reads ~23,000 images and takes several minutes.

In [ ]:
final = assemble.assemble()
final["source"].value_counts()

## 4. The visual comparison measure

Two questions have to be answered before the new images are worth anything:

1. **Are they actually new?** If GBIF or the scrape had simply re-collected
   iNaturalist photos from mirrors, the duplicate rate would be high.
2. **Are they visually different material?** If they were, their dimensions and
   aspect ratios would look like iNaturalist's.

Near-duplicates are found by perceptual hash (Hamming distance ≤ 4), which
catches a re-encoded or resized copy that SHA-256 would miss. Ties are resolved
in favour of the better-identified source.

In [ ]:
report = dedup.comparison_report(final)
report

In [ ]:
# Cross-source duplicates specifically: a scraped image that turns out to be
# an iNaturalist photo we already hold is the failure mode worth naming.
dupes = final[final["duplicate_of"].notna()]
kept_source = final.set_index("photo_id")["source"]
cross = pd.crosstab(dupes["source"],
                    dupes["duplicate_of"].map(kept_source),
                    rownames=["discarded"], colnames=["kept copy from"])
cross

## 5. Class balance

The original dataset ran 500 / 116 — a 4.3:1 ratio. The top-up targets the
starved classes, so the ratio should fall even though the endemics cannot reach
the full +250.

In [ ]:
usable = final[final["ok"].eq(True) & final["duplicate_of"].isna()]

before = inat["label_species"].value_counts()
after = usable["label_species"].value_counts()

balance = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)
balance = balance.sort_values("after")

print(f"before: max {before.max():,}  median {int(before.median()):,}  "
      f"min {before.min():,}  ratio {before.max() / before.min():.1f}:1")
print(f"after : max {after.max():,}  median {int(after.median()):,}  "
      f"min {after.min():,}  ratio {after.max() / after.min():.1f}:1")
balance.tail(10)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
y = np.arange(len(balance))
ax.barh(y, balance["after"], color="#c8d6e5", label="after augmentation")
ax.barh(y, balance["before"], color="#2e86de", label="original (iNaturalist)")
ax.set_yticks(y, balance.index, fontsize=8)
ax.set_xlabel("images")
ax.set_title("Images per species, before and after augmentation")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Spot-check: are the scraped labels right?

The iNaturalist and GBIF rows were identified by naturalists against a taxon
key. The scraped rows were identified by whoever wrote the page the image sat
on, which is not a standard of evidence. This grid is the manual check — any
obviously wrong image should have its `photo_id` added to `REJECTED` below.

In [ ]:
# Shuffle once, then take the first three of each species: `.head(3)` after a
# shuffle is a random sample that keeps every column and copes with species that
# have fewer than three images, which `.sample(3)` per group would not.
sample = (usable[usable["source"] == "web_scrape"]
          .sample(frac=1, random_state=0)
          .groupby("label_species", group_keys=False)
          .head(3)
          .sort_values("label_species"))

fig, axes = plt.subplots(4, 6, figsize=(14, 10))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    try:
        ax.imshow(Image.open(row["local_path"]))
    except Exception:
        pass
    ax.set_title(f"{row['label_species']}\n{row['photo_id']}", fontsize=6)
    ax.axis("off")
for ax in axes.ravel()[len(sample):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# photo_ids judged to be the wrong species in the grid above.
REJECTED: list[str] = []

if REJECTED:
    final.loc[final["photo_id"].isin(REJECTED), ["ok", "reason"]] = [False, "manual_reject"]
    final.to_csv(config.FINAL_CSV, index=False)
    print(f"rejected {len(REJECTED)} images")

## 7. Google Drive

The images are mirrored to Drive with `rclone`, which is idempotent — a
re-run sends only what is missing.

```bash
rclone config                      # once, interactive: remote named 'gdrive'
python -m insects.drive --upload
python -m insects.drive --link
```

Three columns come back. **Train from `drive_path`**, not `drive_url`: reading
thousands of `uc?export=download` links over HTTP trips Drive's abuse threshold,
after which it serves an HTML interstitial instead of a JPEG and the loader sees
a corrupt image — silently, mid-epoch. In Colab, `drive.mount()` and read
`drive_path` as an ordinary file.

In [ ]:
final = pd.read_csv(config.FINAL_CSV)
linked = final["drive_file_id"].notna().sum()
print(f"{linked:,} / {len(final):,} rows carry a Drive id")
final[["photo_id", "source", "drive_path", "drive_url"]].head(3)

## 8. What the final table is

One row per image, 33 classes, four sources.

- **Label**: `label_species`
- **Split key**: `observation_group` — iNaturalist observation id, GBIF
  occurrence key, or (for scraped rows, which have no occurrence) the image
  itself. Splitting on this preserves audit finding S5: several photos of one
  individual must never straddle train and test.
- **Filter before training**: `ok == True` and `duplicate_of.isna()`.
- **Filter before publishing**: `redistributable == True` — this drops the
  scraped images and the NoDerivatives ones.
- **Filter before any commercial use**: `commercial_ok == True`.

In [ ]:
print(f"rows         : {len(final):,}")
print(f"usable       : {(final['ok'].eq(True) & final['duplicate_of'].isna()).sum():,}")
print(f"redistributable: {final['redistributable'].sum():,}")
print(f"commercial_ok  : {final['commercial_ok'].sum():,}")
final.columns.tolist()

## 9. Regenerate the datasheet

`Reports/butterfly_images_datasheet.md` documents the dataset per Gebru et al.
It is regenerated from the table rather than edited, so it cannot drift.

In [ ]:
from insects import datasheet  # noqa: E402

print(datasheet.write(final))